# ChurnIQ — Exploratory Data Analysis

Exploration only. The production pipeline lives in `src/` — this notebook is where
the feature ideas came from, kept in the repo so the reasoning is visible.

Run `python run_pipeline.py` first so the database exists.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import load_config
from features import build_feature_table

pd.set_option("display.max_columns", 60)
cfg = load_config("../config.yaml")
cfg.root = cfg.root  # paths resolve relative to the project root

df = build_feature_table(cfg)
print(df.shape)
df.head()

## 1. How imbalanced is the target?

This decides which metrics are meaningful.

In [ ]:
churn = (df["Churn"] == "Yes")
print(f"churn rate: {churn.mean():.1%}")
print(f"a 'nobody churns' model would score {1 - churn.mean():.1%} accuracy")
print("-> accuracy is not a useful headline metric here")

churn.value_counts().plot(kind="bar", title="Churn distribution", rot=0);

## 2. Contract type

Expected to be the strongest single signal.

In [ ]:
by_contract = (
    df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
)
print(by_contract.round(3))
by_contract.plot(kind="barh", title="Churn rate by contract type")
plt.xlabel("churn rate");

## 3. Tenure

Is the relationship linear? If not, bucketing is justified.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[churn]["tenure"].hist(bins=36, alpha=0.6, label="churned", ax=axes[0])
df[~churn]["tenure"].hist(bins=36, alpha=0.6, label="stayed", ax=axes[0])
axes[0].set_title("Tenure distribution"); axes[0].set_xlabel("months"); axes[0].legend()

rate_by_tenure = df.groupby("tenure")["Churn"].apply(lambda s: (s == "Yes").mean())
rate_by_tenure.plot(ax=axes[1], title="Churn rate by tenure")
axes[1].set_ylabel("churn rate")
plt.tight_layout()

# The curve is steep early then flattens -> non-linear -> tenure_bucket earns its place.

## 4. Payment method and billing

In [ ]:
for col in ["PaymentMethod", "PaperlessBilling", "InternetService"]:
    rates = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
    print(f"\n{col}"); print(rates.round(3).to_string())

## 5. Do bundled services protect against churn?

Motivates `services_count`.

In [ ]:
rate = df.groupby("services_count")["Churn"].apply(lambda s: (s == "Yes").mean())
counts = df["services_count"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
rate.plot(ax=ax, marker="o", label="churn rate")
ax.set_xlabel("number of services subscribed"); ax.set_ylabel("churn rate")
ax.set_title("Churn rate vs. service count")
ax2 = ax.twinx(); counts.plot(ax=ax2, alpha=0.2, kind="bar", color="grey")
ax2.set_ylabel("customers (bars)")
ax.legend();

## 6. Charges

Does the current bill differ from the lifetime average? Motivates `charge_ratio`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[0])
axes[0].set_title("Monthly charges by churn")
df.boxplot(column="charge_ratio", by="Churn", ax=axes[1])
axes[1].set_title("Charge ratio by churn")
plt.suptitle("")
plt.tight_layout()

print(df.groupby("Churn")[["MonthlyCharges", "TotalCharges", "avg_monthly_spend", "charge_ratio"]].mean().round(2))

## 7. Where is the money?

Churn rate alone is the wrong lens — a 40% churn rate among low-value
customers matters less than 20% among high-value ones.

In [ ]:
df["value_12m"] = df["MonthlyCharges"] * 12
lost = df[churn].groupby("Contract")["value_12m"].sum().sort_values(ascending=False)
print(f"total 12-month value of churned customers: ${lost.sum():,.0f}")
lost.plot(kind="barh", title="12-month revenue lost to churn, by contract type")
plt.xlabel("$");

## 8. Correlations among numeric features

Checking for redundancy before modelling.

In [ ]:
num = df.select_dtypes(include=[np.number])
corr = num.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
fig.colorbar(im)
ax.set_title("Numeric feature correlations")
plt.tight_layout()

# tenure / TotalCharges are strongly correlated by construction - expected, and
# tree models handle it. Worth noting for the logistic regression baseline.

## Takeaways carried into `src/features.py`

1. Target is imbalanced (~26%) → report ROC-AUC, PR-AUC, precision and recall, not accuracy.
2. Tenure's effect is non-linear → `tenure_bucket`.
3. Month-to-month contracts dominate churn → `is_month_to_month`.
4. Electronic-check payers churn far more → `has_auto_payment`.
5. Bundled services are protective → `services_count`.
6. Current bill vs. lifetime average carries signal → `charge_ratio`.
7. Revenue concentration matters more than raw churn rate → drives `revenue_at_risk`
   and the threshold tuning in `business.py`.